In [1]:
import os

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
import requests

In [2]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_api_url = "https://api.anthropic.com/v1/"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_api_url)

In [ ]:
MODEL = "gpt-5.6-luna"
TTS_MODEL = "tts-1"

In [4]:
def fetch_website_contents(url: str) -> str | None:
    """Fetches the contents of a website given its URL."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return title + "\n\n" + text

In [5]:
def fetch_wikipedia_page(subject:str) -> str | None:
    url = f"https://en.wikipedia.org/wiki/{subject.lower().strip().replace(' ', '_').replace("-", "_")}"
    return fetch_website_contents(url)

In [6]:
city = "Paris"

In [7]:
page = fetch_wikipedia_page(city)

In [8]:
def summarize_text(text: str, model: str = MODEL) -> str | None:
    system_prompt = """You are a helpful assistant that summarizes text in a very concise (less or equal to 2000 characters), structured and compelling way,
    ignoring text that might be navigation related. 
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Summarize the following text in a very concise (less or equal to 2000 characters), structured and compelling way:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    result = response.choices[0].message.content
    return result[:4096] if result else None

In [9]:
if page:
    summarized_page = summarize_text(page)

In [10]:
def summarize_wikipedia_page(subject: str) -> str | None:
    """Fetches the Wikipedia page for a given subject and summarizes its content."""
    print("TOOL CALL: summarize_wikipedia_page")
    page = fetch_wikipedia_page(subject)
    if not page:
        return None
    summary = summarize_text(page)
    return summary

In [11]:
def find_out_the_main_spoken_language_in_city(city: str, model: str = MODEL) -> str | None:
    system_prompt = """You are a helpful assistant that finds out the main language spoken in a city.
    Respond with the name of the language only, without any additional text."""
    user_prompt = f"What is the main language spoken in {city}?"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content

In [12]:
spoken_language = find_out_the_main_spoken_language_in_city(city)

In [13]:
def translate_text(text: str, target_language: str, model: str = MODEL) -> str | None:
    system_prompt = f"""You are a helpful assistant that translates text into {target_language}.
    Respond with the translated text only, without any additional text.
    The response should contain 2000 characters at most.
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Translate the following text into {target_language}, in 2000 characters or less:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content[:4096] if response.choices[0].message.content else None

In [14]:
if summarized_page and spoken_language:
    translated_summary = translate_text(summarized_page, spoken_language)

In [15]:
def translate_text_into_city_language(text: str, city: str) -> str | None:
    """Translates the given text into the main language spoken in the specified city."""
    print("TOOL CALL: translate_text_into_city_language")
    main_language = find_out_the_main_spoken_language_in_city(city)
    if not main_language:
        return None
    if main_language.lower() == "english":
        return text
    translated_text = translate_text(text, main_language)
    return translated_text

In [16]:
def talker(text: str, model: str = "tts-1") -> bytes | None:
    """Generates speech from the given text using OpenAI's TTS model."""
    response = openai.audio.speech.create(model=model, voice="coral", input=text[:4096], speed=1)
    return response.content

In [17]:
# from IPython.display import Audio, display

# if translated_summary:
#     audio_bytes = talker(translated_summary)
#     display(Audio(audio_bytes, autoplay=True))

In [18]:
def fetch_wikipedia_image_url(subject: str) -> str | None:
    """Fetches the first image URL from the Wikipedia page of the given subject."""
    normalized_subject = subject.lower().strip().replace(" ", "_").replace("-", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{normalized_subject}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    image = response.json().get("originalimage")
    return image["source"] if image else None

In [19]:
def get_function_name_and_description(func) -> tuple[str, str | None]:
    name = func.__name__
    description = func.__doc__ or None
    return (name, description)

In [20]:
def turn_function_into_tool(func, required_parameters):
    name, description = get_function_name_and_description(func)

    function_dictionary = {
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": required_parameters,
            "required": list(required_parameters.keys()),
            "additionalProperties": False,
        },
    }

    return {"type": "function", "function": function_dictionary}


# Parameters

In [21]:
summarize_wikipedia_page_params = {
    "subject": {
        "type": "string",
        "description": "The subject of the Wikipedia page to summarize.",
    }
}

translate_text_into_city_language_params = {
    "text": {
        "type": "string",
        "description": "The text to translate into the main language spoken in the specified city.",
    },
    "city": {
        "type": "string",
        "description": "The city whose main language will be used for translation.",
    },
}

# Tools

In [22]:
summarize_wikipedia_page_tool = turn_function_into_tool(
    summarize_wikipedia_page, summarize_wikipedia_page_params
)
translate_text_into_city_language_tool = turn_function_into_tool(
    translate_text_into_city_language, translate_text_into_city_language_params
)

In [23]:
tools = [
    summarize_wikipedia_page_tool,
    translate_text_into_city_language_tool,
]

In [24]:
tools

[{'type': 'function',
  'function': {'name': 'summarize_wikipedia_page',
   'description': 'Fetches the Wikipedia page for a given subject and summarizes its content.',
   'parameters': {'type': 'object',
    'properties': {'subject': {'type': 'string',
      'description': 'The subject of the Wikipedia page to summarize.'}},
    'required': ['subject'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'translate_text_into_city_language',
   'description': 'Translates the given text into the main language spoken in the specified city.',
   'parameters': {'type': 'object',
    'properties': {'text': {'type': 'string',
      'description': 'The text to translate into the main language spoken in the specified city.'},
     'city': {'type': 'string',
      'description': 'The city whose main language will be used for translation.'}},
    'required': ['text', 'city'],
    'additionalProperties': False}}}]

# Prompts

In [25]:
import json

dict_example = {
    "summary": "If the user provides a city: the summary of the Wikipedia page in English. If the user doesn't provide a city: the message answering normally and finishing by asking them to provide a city.",
    "translated_summary": "If the user provides a city: the summary translated into the main language spoken in the city, without any additional text or explanation. If the user doesn't provide a city: an empty string.",
}

json_example = json.dumps(dict_example, indent=4)

In [26]:
system_prompt = f"""You are a helpful assistant that provides information about cities.
When a user asks for information about a city, you will provide a summary of the Wikipedia page for that city, as well as a translation of that summary into the main language spoken in the city.
When the user doesn't mention a city, answer normally and finish by asking them for a city until they provide one for you to summarize.
You must always respond in JSON format, with the exact structure as follows (exact same keys, values as described), without any additional text or explanation:
{json_example}
"""


In [27]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        tool = globals()[tool_name]
        result = tool(**tool_args)
        responses.append(
            {
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id,
            }
        )
    return responses

In [28]:
RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "city_info",
        "schema": {
            "type": "object",
            "properties": {
                "summary": {"type": "string"},
                "translated_summary": {"type": "string"},
            },
            "required": ["summary", "translated_summary"],
        },
    },
}

In [36]:
def chat(user_message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    print("MESSAGE:", user_message)
    print("HISTORY:", history)
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": user_message}]
    )
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        reasoning_effort="none",
        response_format=RESPONSE_FORMAT,
    )
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_message)
        messages.append(assistant_message)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            reasoning_effort="none",
            response_format=RESPONSE_FORMAT,
        )
    result = response.choices[0].message.content
    dict_result = json.loads(result) if result else {}
    
    summary = dict_result.get("summary", "")
    translated_summary = dict_result.get("translated_summary", "")
    
    history.extend([
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": summary},
    ])
    
    return history, translated_summary, ""

In [30]:
import gradio as gr

c:\Users\MaximeNORVAL\dev\personal\city-traveler\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [31]:
# gr.ChatInterface(fn=chat).launch(inbrowser=True)

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        translated_markdown = gr.Markdown(label="Translated Summary", height=500)
    with gr.Row():
        user_input_textbox = gr.Textbox(
            label="Ask about a city", placeholder="Type your question here...",
        )
    user_input_textbox.submit(
        fn=chat, inputs=[user_input_textbox, chatbot], outputs=[chatbot, translated_markdown, user_input_textbox]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


MESSAGE: Hi there
HISTORY: []
MESSAGE: Can you tell me about Montpellier?
HISTORY: [{'role': 'user', 'content': [{'text': 'Hi there', 'type': 'text'}]}, {'role': 'assistant', 'content': [{'text': 'Hi there! How can I help you? Please provide a city for me to summarize.', 'type': 'text'}]}]
TOOL CALL: summarize_wikipedia_page
TOOL CALL: translate_text_into_city_language
